# Tutorial 1: Active inference from scratch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infer-actively/pymdp/blob/main/docs/notebooks/active_inference_from_scratch.ipynb)

*Author: Conor Heins*

This tutorial guides you through the construction of an active inference agent "from scratch" in a simple grid-world environment. This tutorial is designed to walk you through the various mathematical operations required for running active inference in discrete state spaces, as well as introduce you to the way discrete probability distributions are represented in `pymdp` (through the use of numpy vectors and matrices). We also show you how to compute the expected free energy in terms of risk and ambiguity, and how to use it to plan sequences of actions.

By the end of this tutorial, you should be able to write simple active inference models from scratch using the main components of any POMDP generative model: the `A`, `B`, `C`, and `D` arrays, as well as be able to write the active inference loop for an agent engaged in a perception-action exchange with its environment.

In [ ]:
# pip install inferactively-pymdp

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Define some auxiliary functions
for plotting

In [ ]:
def plot_likelihood(matrix, xlabels = list(range(9)), ylabels = list(range(9)), title_str = "Likelihood distribution (A)"):
    """
    Plots a 2-D likelihood matrix as a heatmap
    """
    
    if not np.isclose(matrix.sum(axis=0), 1.0).all():
        raise ValueError("Distribution not column-normalized! Please normalize (ensure matrix.sum(axis=0) == 1.0 for all columns!)")
    
    plt.figure(figsize = (6,6))
    sns.heatmap(matrix, xticklabels = xlabels, yticklabels = ylabels, cmap = 'gray', cbar = False, vmin = 0.0, vmax = 1.0)
    plt.title(title_str)
    plt.show()
    
def plot_grid(grid_locations, num_x = 3, num_y = 3):
    """
    Plots the spatial coordinates of GridWorld as a heatmap, with each (X, Y) coordinate
    labeled with its linear index (its `state id`)
    """
    
    grid_heatmap = np.zeros((num_x, num_y))
    for linear_idx, location in enumerate(grid_locations):
        y, x = location
        grid_heatmap[y, x] = linear_idx
    sns.set(font_scale=1.5)
    sns.heatmap(grid_heatmap, annot=True, cbar = False, fmt='.0f', cmap='crest')
        
def plot_point_on_grid(state_vector, grid_locations):
    """
    Plots the current location of the agent on the grid world
    """
    state_index = np.where(state_vector)[0][0]
    y, x = grid_locations[state_index]
    grid_heatmap = np.zeros((3,3))
    grid_heatmap[y,x] = 1.0
    sns.heatmap(grid_heatmap, cbar = False, fmt='.0f')
    
def plot_beliefs(belief_dist, title_str=""):
    """
    Plot a categorical distribution or belief distribution, stored in the 1-D numpy vector `belief_dist`
    """
    
    if not np.isclose(belief_dist.sum(), 1.0):
        raise ValueError("Distribution not normalized! Please normalize")
    
    plt.grid(zorder=0)
    plt.bar(range(belief_dist.shape[0]), belief_dist, color='r', zorder=3)
    plt.xticks(range(belief_dist.shape[0]))
    plt.title(title_str)
    plt.show()
    

# Categorical distributions

In [ ]:
from pymdp import utils

In [ ]:
my_categorical = np.random.rand(3)
# my_categorical = np.array([0.5, 0.3, 0.8]) # can write other numbers
my_categorical = utils.norm_dist(my_categorical) # normalize the distribution so it integrates to 1.0

print(my_categorical.reshape(-1, 1)) # we reshape it to display as a column vector
print(f'Integral of the distribution: {round(my_categorical.sum(), 2)}')

In [ ]:
# We can sample a discrete outcome from this distribution with `sample()` from the `utils` module

sampled_outcome = utils.sample(my_categorical)
print(f'Sampled outcome: {sampled_outcome}')

In [ ]:
# Now plot the beliefs using `plot_beliefs()`, defined in the beginning

plot_beliefs(my_categorical, title_str = "A random (unconditional) categorical distribution")

In [ ]:
# Now moving on to *conditional* categorical distributions or likelihoods 
# -- i.e. how we represent the distribution of one discrete random variable X,
# conditioned on the settings of another discrete random variable Y.
# Mathematically: P(X|Y)

# Initialize it with random numbers
p_x_given_y = np.random.rand(3, 4)
print(p_x_given_y.round(3))

In [ ]:
# Normalize it
p_x_given_y = utils.norm_dist(p_x_given_y)
print(p_x_given_y)

In [ ]:
# Print the first column (i.e. P(X|Y) = 0) and show it's a proper categorical distribution
print(p_x_given_y[:,0].reshape(-1,1))
print(f'Integral of P(X|Y=0): {p_x_given_y[:,0].sum()}')

In [ ]:
# So column `i` of the matrix `p_x_given_y` represents the conditional probability of X, 
# given the i-th level of the random variable Y -- i.e. P(X|Y = i)

# Next: taking expectations of random variables using matrix-vector products

""" Create a P(Y) and P(X|Y) using the same numbers from the slides """

p_y = np.array([0.75, 0.25]) # already normalized no need to `utils.norm_dist()` it

# columns also already normalized
p_x_given_y = np.array([[0.6, 0.5],
                        [0.15, 0.41],
                        [0.25, 0.09]])

print(p_y.round(3).reshape(-1,1))
print(p_x_given_y.round(3))

In [ ]:
# Calculate expected value of X, given our belief about Y, P(Y) using a simple vector-matric product, of the form Ax

# first method of dot product: numpy array
E_x_wrt_y = p_x_given_y.dot(p_y)

# second method of dot product: function np.dot with two arguments
# E_x_wrt_y = np.dot(p_x_given_y, p_y)

In [ ]:
# Now print and verify the result is a proper categorical distribution (integrates to 1.0)

print(E_x_wrt_y)
print(f'Integral: {E_x_wrt_y.sum().round(3)}')

### Grid-world environment

In [ ]:
import itertools

""" Create the grid locations in the form of (Y, X) tuples """
grid_locations = list(itertools.product(range(3), repeat = 2))
print(grid_locations)

In [ ]:
plot_grid(grid_locations)

basically, the likelihood matrix is (X, Y) where each X is an observation and each Y is a state.
A transition matrix has each actions (eg up, down, left, right = 9x9 matrix) and models how an action given a state (X) transitions into another state (Y).

# Building the generative model: A, B, C, and D

## 1. The A matrix, P(o | s)
The generative model's "prior beliefs" about how hidden states relate to observations

In [ ]:
""" Create variables for storing the dimensionalities of the hidden states and observations """

n_states = len(grid_locations)
n_observations = len(grid_locations)

print(f'Dimensionality of hidden states: {n_states}')
print(f'Dimensionality of observations: {n_observations}')

In [ ]:
""" Create the A matrix """

A = np.zeros( (n_states, n_observations))

In [ ]:
""" Create an unambiguous or 'noise-less' mapping between hidden states and observations """

np.fill_diagonal(A, 1.0)

# alternative:
# A = np.eye(n_observations, n_states)

Likelihood matrices show relationships between indices of the state space, which are indexed linearly. 
Remember the (Y, X) coordinates of the grid world don't necessarily map neatly onto the locations in 
the matrix here, which has "unwrapped" the Grid World's (Y, X) locations into columns/rows


In [ ]:
plot_likelihood(A, title_str = "A matrix or $P(o|s)$") 

How could we introduce uncertainty into this matrix? How do we introduce ambiguity or uncertainty into the agent's model of the world?

In [ ]:
""" Remind yourself of the mapping between linear indices (0 through 8) and grid locations (Y, X) """
plot_grid(grid_locations)

In [ ]:
A_noisy = A.copy()

# Hard code some probabilities.

# The probability of seeing yourself at location 0, given you're in location 0, is 1/3 -- P(o == 0 | s == 0) = 0.333...
A_noisy[0,0] = 1/3.0 # Location (0,0)

# The probability of seeing yourself in location 1, given you're in location 0, is 1/3 -- P(o == 1 | s == 0) = 0.333...
A_noisy[1,0] = 1/3.0 # Location (1,0): one step right from state 0

A_noisy[3,0] = 1/3.0 # Location (3,0): one step down from state 0

In [ ]:
plot_likelihood(A_noisy, title_str='modified A matrix where location (0,0) is "blurry"')